# 25 — Satellite Plume Fingerprinting

April 2025 replacement.

In [ ]:

SITE_ID = "NHV"
SITE_NAME = "Newhaven ERF"
LAT = 50.80208
LON = 0.04961
DATE_FROM = "2025-04-02"
DATE_TO   = "2025-04-07"
SCENE_CATALOG_CSV = "outputs/s5p/NHV_no2_offl_scene_catalog.csv"
OUTPUT_DIR = "outputs/plume_fingerprint"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": start_date, "end_date": end_date, "hourly": hourly_vars, "timezone": "UTC"}
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True); CACHE_DIR=Path(OUTPUT_DIR)/"weather_cache"; CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:

scene_df,scene_meta=safe_read_csv(SCENE_CATALOG_CSV)
if not scene_df.empty and "start_utc" in scene_df.columns:
    scene_df["start_utc"]=pd.to_datetime(scene_df["start_utc"],utc=True,errors="coerce"); scene_df=scene_df.dropna(subset=["start_utc"]).copy()
    if not scene_df.empty: scene_df["date"]=scene_df["start_utc"].dt.date.astype("string")
if not scene_df.empty and "start_utc" in scene_df.columns: scene_df=scene_df.sort_values("start_utc").reset_index(drop=True)
j=fetch_weather_cached(CACHE_DIR,LAT,LON,DATE_FROM,DATE_TO,"wind_speed_10m,wind_direction_10m,cloud_cover"); h=j.get("hourly",{})
wx=pd.DataFrame({"time_utc":pd.to_datetime(h.get("time",[]),utc=True,errors="coerce"),"wind_speed_ms":h.get("wind_speed_10m",[]),"wind_dir_deg":h.get("wind_direction_10m",[]),"cloud_cover_pct":h.get("cloud_cover",[])})
def nearest_hour(ts,w):
    if w.empty or pd.isna(ts): return pd.Series({"wind_speed_ms":np.nan,"wind_dir_deg":np.nan,"cloud_cover_pct":np.nan})
    idx=(w["time_utc"]-ts).abs().idxmin(); return w.loc[idx]
rows=[]
if not scene_df.empty and "date" in scene_df.columns:
    for _,r in scene_df.iterrows():
        w=nearest_hour(r["start_utc"],wx); wind_speed=pd.to_numeric(w.get("wind_speed_ms",np.nan),errors="coerce"); cloud_cover=pd.to_numeric(w.get("cloud_cover_pct",np.nan),errors="coerce")
        rows.append({"product_id":r.get("product_id"),"date":r.get("date"),"start_utc":r.get("start_utc"),"wind_speed_ms":wind_speed,"wind_dir_deg":pd.to_numeric(w.get("wind_dir_deg",np.nan),errors="coerce"),"cloud_cover_pct":cloud_cover,"fingerprint_score":max(0.0,5.0-(wind_speed if pd.notna(wind_speed) else 5.0))+max(0.0,100.0-(cloud_cover if pd.notna(cloud_cover) else 100.0))/25.0})
fp=pd.DataFrame(rows); fp.to_csv(Path(OUTPUT_DIR)/"plume_fingerprint_scene_scores.csv",index=False)
if not fp.empty and "date" in fp.columns:
    daily=fp.groupby("date",dropna=False).agg(scenes=("product_id","count"),mean_wind_speed_ms=("wind_speed_ms","mean"),mean_cloud_cover_pct=("cloud_cover_pct","mean"),fingerprint_score=("fingerprint_score","mean")).reset_index()
else:
    daily=pd.DataFrame(columns=["date","scenes","mean_wind_speed_ms","mean_cloud_cover_pct","fingerprint_score"])
daily.to_csv(Path(OUTPUT_DIR)/"plume_fingerprint_daily_summary.csv",index=False); display(daily.head(20))


In [ ]:

fig,ax=plt.subplots(figsize=(10,5))
if not daily.empty:
    x=pd.to_datetime(daily["date"]); ax.plot(x,daily["fingerprint_score"],marker="o",label="fingerprint_score"); ax.plot(x,daily["scenes"],marker="o",label="scene_count")
ax.set_title("Satellite plume fingerprinting screen"); ax.set_xlabel("Date"); ax.set_ylabel("Score")
if not daily.empty: ax.legend()
fig.autofmt_xdate(); fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/"plume_fingerprint_plot.png"; fig.savefig(plot_path,dpi=150); plt.show()
(Path(OUTPUT_DIR)/"plume_fingerprint_manifest.json").write_text(json.dumps({"created_utc":datetime.now(timezone.utc).isoformat(),"input_scene_catalog":scene_meta,"rows_scene_df":int(len(scene_df)),"rows_fp":int(len(fp)),"rows_daily":int(len(daily))},indent=2),encoding="utf-8")
print("Saved:",plot_path)
